### 데이터 조회를 위한 도구 설치 및 API키 연결

In [58]:
import os
import io
import zipfile
import xml.etree.ElementTree as ET
import json

import pandas as pd
import requests
from dotenv import load_dotenv

# 프로젝트 루트의 .env 파일을 읽습니다.
load_dotenv("/Users/anna/bulsae/.env")

# API 키는 변수에만 저장하고 화면에 출력하지 않습니다.
dart_api_key = os.getenv("DART_API_KEY")

if not dart_api_key:
    raise ValueError(".env 파일에 DART_API_KEY가 있는지 확인해 주세요.")

print("API 키를 확인했습니다.")

API 키를 확인했습니다.


### 개황데이터 전처리

In [59]:
# 사전에 설정한 기업 불러오기

file_path = "/Users/anna/bulsae/data/shared/companies.jsonl"

with open(file_path, encoding="utf-8") as file:
    companies = [json.loads(line) for line in file if line.strip()]

companies_df = pd.DataFrame(companies)
display(companies_df)

,company_id,input_name,legal_name,dart_corp_code
0,COMPANY-INNISFREE,이니스프리,주식회사 이니스프리,00812353
1,COMPANY-ESPOIR,에스쁘아,주식회사 에스쁘아,01061150
2,COMPANY-AMOREPACIFIC,아모레퍼시픽,주식회사 아모레퍼시픽,00583424
3,COMPANY-ETUDE,에뛰드,주식회사 에뛰드,00183808
4,COMPANY-PACIFICTECH,퍼시픽테크,주식회사 퍼시픽테크,01784251
5,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,주식회사 아모스프로페셔널,00144456
6,COMPANY-COSRX,코스알엑스,주식회사 코스알엑스,01364491


In [60]:
# 사전에 설정한 기업 목록 한정 Dart API에서 제공하는 데이터 불러오기 
url = "https://opendart.fss.or.kr/api/company.json"

results = []

for company in companies:
    response = requests.get(
        url,
        params={
            "crtfc_key": dart_api_key,
            "corp_code": company["dart_corp_code"],
        },
        timeout=30,
    )

    if response.status_code != 200:
        print(f'{company["input_name"]}: HTTP 오류 {response.status_code}')
        continue

    data = response.json()

    # Open DART 응답 코드가 000이면 정상입니다.
    if data.get("status") != "000":
        print(
            f'{company["input_name"]}: '
            f'{data.get("status")} - {data.get("message")}'
        )
        continue

    # 우리 프로젝트의 기업 ID도 함께 넣어 나중에 연결할 수 있게 합니다.
    data["company_id"] = company["company_id"]
    data["input_name"] = company["input_name"]
    results.append(data)

print(f"조회 성공: {len(results)}개 / 전체 {len(companies)}개")

조회 성공: 7개 / 전체 7개


In [61]:
result_df = pd.DataFrame(results)

# 요청한 대로 최대 10개까지만 표시합니다.
display(result_df.head(3))

,status,message,corp_code,corp_name,corp_name_eng,stock_name,stock_code,ceo_nm,corp_cls,jurir_no,...,adres,hm_url,ir_url,phn_no,fax_no,induty_code,est_dt,acc_mt,company_id,input_name
0,000,정상,00812353,(주)이니스프리,Innisfree Corporation,이니스프리,,최민정,E,1101114216522,...,서울특별시 용산구 한강대로 100 (한강로2가),www.innisfree.co.kr,,02-2186-7110,02-6040-7108,46443,20091110,12,COMPANY-INNISFREE,이니스프리
1,000,정상,01061150,주식회사 에스쁘아,ESPOIR CORPORATION,에스쁘아,,이연정,E,1101115605013,...,서울특별시 용산구 한강대로 100 (한강로2가),,,02-862-2604,02-6040-2700,46443,20150102,12,COMPANY-ESPOIR,에스쁘아
2,000,정상,00583424,(주)아모레퍼시픽,AMOREPACIFIC CORP.,아모레퍼시픽,090430,"서경배, 김승환",Y,1101113471036,...,서울특별시 용산구 한강대로 100 (한강로2가),www.apgroup.com,,02-6040-5114,02-6040-6349,20423,20060601,12,COMPANY-AMOREPACIFIC,아모레퍼시픽


In [62]:
# 날짜를 YYYYMMDD에서 YYYY-MM-DD로 바꾸는 함수입니다.
def format_established_date(value):
    # 값이 없거나 빈 문자열이면 None으로 남깁니다.
    if pd.isna(value) or not str(value).strip():
        return None

    # API에서 받은 날짜를 변환합니다.
    date = pd.to_datetime(str(value).strip(), format="%Y%m%d", errors="coerce")

    # 날짜 형식이 잘못되었으면 임의로 변환하지 않습니다.
    if pd.isna(date):
        return None

    return date.strftime("%Y-%m-%d")

In [63]:
# 법인구분에서 확인 가능한 상장 상태만 변환합니다.
def get_listing_status(corp_cls):
    # Y: 유가증권, K: 코스닥, N: 코넥스
    if corp_cls in ["Y", "K", "N"]:
        return "상장"

    # E는 '기타'이므로 비상장 또는 상장폐지로 단정하지 않습니다.
    return None

In [64]:
# 기업개황 결과를 정리해서 저장할 리스트입니다.
company_overview_rows = []

for _, row in result_df.iterrows():
    company_overview_rows.append({
        # 기업 식별에 사용하는 우리 프로젝트의 공통 ID입니다.
        "company_id": row["company_id"],

        # 작업용 표에서 알아보기 쉽도록 기업명을 남깁니다.
        "input_name": row["input_name"],

        # 업종코드는 분류명을 확인하기 위한 참고값입니다.
        # 코드 자체를 industry 필드에 넣지는 않습니다.
        "industry_code": row.get("induty_code"),

        # 업종명은 분류명을 확인한 뒤 입력합니다.
        "industry": None,

        # API의 법인구분 원본값도 함께 보관합니다.
        "corp_cls": row.get("corp_cls"),

        # 명확하게 상장으로 확인되는 경우만 변환합니다.
        "listing_status": get_listing_status(row.get("corp_cls")),

        # 설립일을 우리 스키마의 날짜 형식으로 바꿉니다.
        "established_date": format_established_date(row.get("est_dt")),
    })

# 리스트를 표로 변환합니다.
overview_df = pd.DataFrame(company_overview_rows)

display(overview_df.head(20))

,company_id,input_name,industry_code,industry,corp_cls,listing_status,established_date
0,COMPANY-INNISFREE,이니스프리,46443,None,E,NaN,2009-11-10
1,COMPANY-ESPOIR,에스쁘아,46443,None,E,NaN,2015-01-02
2,COMPANY-AMOREPACIFIC,아모레퍼시픽,20423,None,Y,상장,2006-06-01
3,COMPANY-ETUDE,에뛰드,46443,None,E,NaN,1966-02-19
4,COMPANY-PACIFICTECH,퍼시픽테크,28519,None,E,NaN,2023-07-27
5,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,46443,None,E,NaN,1976-01-24
6,COMPANY-COSRX,코스알엑스,20423,None,E,NaN,2013-12-11


In [65]:
pd.set_option("display.max_colwidth", None)

### 재무제표 API 데이터 전처리

In [66]:
# 아모레퍼시픽의 2025년 개별 재무제표 조회
url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

params = {
    "crtfc_key": dart_api_key,   # .env에서 불러온 API 키
    "corp_code": "00583424",    # 아모레퍼시픽 DART 고유번호
    "bsns_year": "2025",        # 사업연도
    "reprt_code": "11011",      # 사업보고서
    "fs_div": "OFS",            # 개별 재무제표
}

response = requests.get(url, params=params, timeout=30)

if response.status_code != 200:
    print(f"HTTP 오류: {response.status_code}")
else:
    data = response.json()

    if data.get("status") == "000":
        # API가 반환한 재무제표 항목을 표로 변환
        finance_df = pd.DataFrame(data.get("list", []))

        print(f"조회된 재무제표 항목: {len(finance_df)}개")
        print("제공되는 컬럼:", finance_df.columns.tolist())
        display(finance_df.head(10))
    else:
        print(f'조회 실패: {data.get("status")} - {data.get("message")}')

조회된 재무제표 항목: 173개
제공되는 컬럼: ['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'sj_div', 'sj_nm', 'account_id', 'account_nm', 'account_detail', 'thstrm_nm', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_amount', 'bfefrmtrm_nm', 'bfefrmtrm_amount', 'ord', 'currency', 'thstrm_add_amount']


,rcept_no,reprt_code,bsns_year,corp_code,sj_div,sj_nm,account_id,account_nm,account_detail,thstrm_nm,thstrm_amount,frmtrm_nm,frmtrm_amount,bfefrmtrm_nm,bfefrmtrm_amount,ord,currency,thstrm_add_amount
0,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_Assets,자산총계,-,제 20 기,5625244226847,제 19 기,5378005124117,제 18 기,5374183609328,7,KRW,NaN
1,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_CurrentAssets,유동자산,-,제 20 기,1317972747968,제 19 기,1102807414388,제 18 기,1563590026012,8,KRW,NaN
2,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_CashAndCashEquivalents,현금및현금성자산,-,제 20 기,162108994546,제 19 기,92179300712,제 18 기,235060688271,9,KRW,NaN
3,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_CurrentFinancialAssetsAtAmortisedCost,상각후원가측정금융자산,-,제 20 기,37011000000,제 19 기,30000000000,제 18 기,58630000000,10,KRW,NaN
4,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_CurrentFinancialAssetsAtFairValueThroughProfitOrLoss,당기손익-공정가치측정금융자산,-,제 20 기,424354909532,제 19 기,229809549698,제 18 기,549634663158,11,KRW,NaN
5,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_CurrentTradeReceivables,매출채권,-,제 20 기,314541910332,제 19 기,400464990095,제 18 기,363532853995,12,KRW,NaN
6,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_Inventories,재고자산,-,제 20 기,244049985141,제 19 기,246415173233,제 18 기,237401032140,13,KRW,NaN
7,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_OtherCurrentAssets,기타유동자산,-,제 20 기,26329358374,제 19 기,24832177828,제 18 기,24760700782,14,KRW,NaN
8,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_OtherCurrentReceivables,기타수취채권,-,제 20 기,40326590043,제 19 기,48856819132,제 18 기,40775874287,15,KRW,NaN
9,20260318000785,11011,2025,00583424,BS,재무상태표,ifrs-full_ShorttermDepositsNotClassifiedAsCashEquivalents,금융기관예치금,-,제 20 기,69250000000,제 19 기,30249403690,제 18 기,53794213379,16,KRW,NaN


In [67]:
url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

# 찾고 싶은 재무 항목
target_names = [
    "매출액",
    "매출",
    "수익(매출액)",
    "영업수익",
    "영업이익",
    "영업이익(손실)",
]

financial_items = []
query_status = []

for company in companies:
    response = requests.get(
        url,
        params={
            "crtfc_key": dart_api_key,
            "corp_code": company["dart_corp_code"],
            "bsns_year": "2025",
            "reprt_code": "11011",  # 사업보고서: 연간
            "fs_div": "OFS",        # 개별 재무제표
        },
        timeout=30,
    )

    if response.status_code != 200:
        query_status.append({
            "기업명": company["input_name"],
            "조회 결과": f"HTTP 오류 {response.status_code}",
        })
        continue

    data = response.json()

    if data.get("status") != "000":
        query_status.append({
            "기업명": company["input_name"],
            "조회 결과": f'{data.get("status")} - {data.get("message")}',
        })
        continue

    # 전체 항목 중 원하는 계정명만 골라 담기
    matched = 0
    for item in data.get("list", []):
        if item.get("account_nm") in target_names:
            financial_items.append({
                "company_id": company["company_id"],
                "기업명": company["input_name"],
                "사업연도": item.get("bsns_year"),
                "보고서": "사업보고서(연간)",
                "재무제표": item.get("sj_nm"),
                "항목": item.get("account_nm"),
                "당기 금액": item.get("thstrm_amount"),
                "접수번호": item.get("rcept_no"),
            })
            matched += 1

    query_status.append({
        "기업명": company["input_name"],
        "조회 결과": f"성공 · 선택된 항목 {matched}개",
    })

# 기업별 조회 상태
print("기업별 조회 결과")
display(pd.DataFrame(query_status))

# 매출·영업이익 조회 결과
financial_df = pd.DataFrame(financial_items)

print("매출·영업이익 항목")
if financial_df.empty:
    print("선택된 항목이 없습니다.")
else:
    display(financial_df.head(10))

기업별 조회 결과


,기업명,조회 결과
0,이니스프리,013 - 조회된 데이타가 없습니다.
1,에스쁘아,013 - 조회된 데이타가 없습니다.
2,아모레퍼시픽,성공 · 선택된 항목 2개
3,에뛰드,013 - 조회된 데이타가 없습니다.
4,퍼시픽테크,013 - 조회된 데이타가 없습니다.
5,아모스프로페셔널,013 - 조회된 데이타가 없습니다.
6,코스알엑스,013 - 조회된 데이타가 없습니다.


매출·영업이익 항목


,company_id,기업명,사업연도,보고서,재무제표,항목,당기 금액,접수번호
0,COMPANY-AMOREPACIFIC,아모레퍼시픽,2025,사업보고서(연간),포괄손익계산서,영업이익,212514854678,20260318000785
1,COMPANY-AMOREPACIFIC,아모레퍼시픽,2025,사업보고서(연간),포괄손익계산서,매출액,2653725793231,20260318000785


아모레퍼시픽 데이터만 존재하여 재무재표 API를 사용하지않고 보고서 데이터를 사용할 예정

### 공시 서류 원본 파일 API 데이터 전처리

In [68]:
# 7개 기업의 2025년 감사보고서 목록 조회
url = "https://opendart.fss.or.kr/api/list.json"

reports = []

for company in companies:
    page = 1

    while True:
        response = requests.get(
            url,
            params={
                "crtfc_key": dart_api_key,
                "corp_code": company["dart_corp_code"],
                "bgn_de": "20260101",
                "end_de": "20261104",
                "last_reprt_at": "Y",
                "page_count": 100,
                "page_no": page,
            },
            timeout=30,
        )

        if response.status_code != 200:
            print(f'{company["input_name"]}: HTTP 오류 {response.status_code}')
            break

        data = response.json()

        if data.get("status") != "000":
            print(
                f'{company["input_name"]}: '
                f'{data.get("status")} - {data.get("message")}'
            )
            break

        for item in data.get("list", []):
            title = item.get("report_nm", "")

            if "감사보고서" in title and "2025.12" in title:
                reports.append({
                    "company_id": company["company_id"],
                    "기업명": company["input_name"],
                    "보고서명": title,
                    "접수번호": item["rcept_no"],
                    "접수일": item.get("rcept_dt"),
                })

        if page >= int(data.get("total_page", 1)):
            break

        page += 1

print(f"찾은 감사보고서: {len(reports)}건")
display(pd.DataFrame(reports).head(10))

찾은 감사보고서: 5건


,company_id,기업명,보고서명,접수번호,접수일
0,COMPANY-INNISFREE,이니스프리,감사보고서 (2025.12),20260401003769,20260401
1,COMPANY-ESPOIR,에스쁘아,감사보고서 (2025.12),20260326001214,20260326
2,COMPANY-ETUDE,에뛰드,감사보고서 (2025.12),20260331004546,20260331
3,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,감사보고서 (2025.12),20260326000513,20260326
4,COMPANY-COSRX,코스알엑스,감사보고서 (2025.12),20260402000034,20260402


In [69]:
# 앞에서 찾은 감사보고서 중 첫 번째 보고서
if not reports:
    print("조회된 감사보고서가 없습니다. 보고서 목록 조회 셀을 먼저 실행해 주세요.")
else:
    report = reports[0]

    print("기업명:", report["기업명"])
    print("보고서명:", report["보고서명"])
    print("접수번호:", report["접수번호"])

    # 접수번호로 공시 원문 요청
    response = requests.get(
        "https://opendart.fss.or.kr/api/document.xml",
        params={
            "crtfc_key": dart_api_key,
            "rcept_no": report["접수번호"],
        },
        timeout=60,
    )

    if response.status_code != 200:
        print("HTTP 오류:", response.status_code)

    elif not zipfile.is_zipfile(io.BytesIO(response.content)):
        # 정상적으로 ZIP이 오지 않았다면 Open DART 오류 메시지 확인
        try:
            error = ET.fromstring(response.content)
            print("조회 실패:", error.findtext("message"))
        except ET.ParseError:
            print("예상하지 못한 응답을 받았습니다.")

    else:
        with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
            file_names = archive.namelist()

            print("\nZIP 안의 파일 목록:")
            for name in file_names:
                print("-", name)

기업명: 이니스프리
보고서명: 감사보고서 (2025.12)
접수번호: 20260401003769

ZIP 안의 파일 목록:
- 20260401003769_00760.xml


In [70]:
from html.parser import HTMLParser

# XML의 태그를 제외하고 글자만 모으는 도구
class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_data(self, data):
        text = data.strip()
        if text:
            self.parts.append(text)


# 앞 셀에서 받은 ZIP을 다시 열어 XML 읽기
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    xml_name = file_names[0]
    xml_content = archive.read(xml_name).decode("utf-8", errors="replace")

# XML에서 텍스트만 꺼내기
extractor = TextExtractor()
extractor.feed(xml_content)
report_text = " ".join(extractor.parts)

# 찾고 싶은 단어가 나오는 위치와 주변 내용 확인
for keyword in ["매출액", "영업이익"]:
    print(f"\n{'=' * 20} {keyword} {'=' * 20}")

    position = 0
    found = 0

    while found < 3:  # 단어별 최대 3곳 확인
        index = report_text.find(keyword, position)

        if index == -1:
            break

        start = max(0, index - 150)
        end = min(len(report_text), index + 350)

        print(f"\n[{found + 1}번째 발견]")
        print(report_text[start:end])

        position = index + len(keyword)
        found += 1

    if found == 0:
        print("해당 단어를 찾지 못했습니다.")


==================== 매출액 ====================

[1번째 발견]
괄 손 익 계 산 서 제 17 기 : 2025년 1월 1일부터 2025년 12월 31일까지 제 16 기 : 2024년 1월 1일부터 2024년 12월 31일까지 주식회사 이니스프리 (단위 : 원) 과             목 주석 제 17 (당) 기 제 16(전) 기 매출액 19,28 209,848,553,422 224,608,927,934 매출원가 20,28 84,193,020,990 86,582,883,635 매출총이익 125,655,532,432 138,026,044,299 판매비와관리비 20,21,28 112,306,620,701 136,390,674,770 영업이익(손실) 13,348,911,731 1,635,369,529 금융수익 4,22 10,810,508,267 17,971,829,946 금융비용 10,22,28 1,827,254,151 5,152,968,982 기타영업외손익 4,23,28 (1,226,642,922) 2,640,526,938 법인세비용차감전순이익(손실

[2번째 발견]
개월 이하인 리스이며, 소액리스자산은 소액의 사무실비품 등으로 구성되어 있습니다. ① 변동리스료일부 부동산 리스는 매장에서 생기는 매출에 연동되는 변동리스료 조건을 포함합니다. 개별 매장의 경우, 리스료의 최대 100%까지 변동리스료 지급 조건을 따르며 리스료 산정시 매출액에 적용되는 비율 또한 그 범위가 넓습니다. 변동리스료를 지급하는 계약조건은 신규 매장의 고정원가 최소화를 포함하여 다양한 이유로 사용됩니다. 매출에 연동하는 변동리스료는 변동리스료를 유발하는 조건이 생기는 기간에 당기손익으로 인식합니다. ② 연장선택권 및 종료선택권 회사 전체에 걸쳐 다수의 부동산 및 시설장치 리스계약에 연장선택권 및 종료선택권을 포함하고 있습니다. 이러한 조건들은 계약 관리 측면에서 운영상의 유연성을 극대화하기 위해 사용됩니다. 보유하고 있는 대부분의 연장선택권 및 종료선택권

In [71]:
from html.parser import HTMLParser

class TableExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tables = []
        self.table = None
        self.row = None
        self.cell = None

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()

        if tag == "table":
            self.table = []
        elif tag == "tr" and self.table is not None:
            self.row = []
        elif tag in ("td", "th") and self.row is not None:
            self.cell = []

    def handle_data(self, data):
        if self.cell is not None:
            self.cell.append(data.strip())

    def handle_endtag(self, tag):
        tag = tag.lower()

        if tag in ("td", "th") and self.cell is not None:
            self.row.append(" ".join(part for part in self.cell if part))
            self.cell = None

        elif tag == "tr" and self.row is not None:
            if self.row:
                self.table.append(self.row)
            self.row = None

        elif tag == "table" and self.table is not None:
            self.tables.append(self.table)
            self.table = None


# 앞에서 읽은 xml_content를 표 단위로 분리
extractor = TableExtractor()
extractor.feed(xml_content)

print(f"보고서에서 찾은 표: {len(extractor.tables)}개")

# 매출액이나 영업이익이 들어간 행을 찾기
matches = []

for table_number, table in enumerate(extractor.tables, start=1):
    for row_number, row in enumerate(table, start=1):
        row_text = " ".join(row)

        if "매출액" in row_text or "영업이익" in row_text:
            matches.append({
                "표 번호": table_number,
                "행 번호": row_number,
                "행 내용": row,
            })

print(f"관련 행: {len(matches)}개")
display(pd.DataFrame(matches).head(10))

보고서에서 찾은 표: 171개
관련 행: 2개


,표 번호,행 번호,행 내용
0,15,2,"[매출액, 19,28, , 209,848,553,422, , 224,608,927,934]"
1,15,6,"[영업이익(손실), , , 13,348,911,731, , 1,635,369,529]"


In [72]:
# 매출액과 영업이익이 함께 있는 표 찾기
for table_number, table in enumerate(extractor.tables, start=1):
    table_text = " ".join(" ".join(row) for row in table)

    if "매출액" in table_text and "영업이익" in table_text:
        print(f"\n===== {table_number}번 표 =====")

        for row in table:
            print(row)


===== 15번 표 =====
['과             목', '주석', '제 17 (당) 기', '제 16(전) 기']
['매출액', '19,28', '', '209,848,553,422', '', '224,608,927,934']
['매출원가', '20,28', '', '84,193,020,990', '', '86,582,883,635']
['매출총이익', '', '', '125,655,532,432', '', '138,026,044,299']
['판매비와관리비', '20,21,28', '112,306,620,701', '', '136,390,674,770', '']
['영업이익(손실)', '', '', '13,348,911,731', '', '1,635,369,529']
['금융수익', '4,22', '10,810,508,267', '', '17,971,829,946', '']
['금융비용', '10,22,28', '1,827,254,151', '', '5,152,968,982', '']
['기타영업외손익', '4,23,28', '(1,226,642,922)', '', '2,640,526,938', '']
['법인세비용차감전순이익(손실)', '', '', '21,105,522,925', '', '17,094,757,431']
['법인세비용(수익)', '18', '5,248,462,676', '', '2,133,266,598', '']
['당기순이익', '', '', '15,857,060,249', '', '14,961,490,833']
['기타포괄손익:', '', '', '485,846,732', '', '(1,360,009,269)']
['당기손익으로 재분류되지 않는 항목', '', '', '', '', '']
['순확정급여부채의 재측정요소', '15,17,18', '485,846,732', '', '(1,360,009,269)', '']
['당기총포괄이익', '', '', '16,342,906,981', '', '13,601,481,564']


In [73]:
# 화면에서 확인한 15번 표
income_table = extractor.tables[14]  # 파이썬에서는 첫 번째가 0번

# 괄호로 표시된 손실도 음수로 변환
def parse_amount(value):
    value = value.strip().replace(",", "")

    if value.startswith("(") and value.endswith(")"):
        return -int(value[1:-1])

    return int(value)


revenue = None
operating_profit = None

for row in income_table:
    if not row:
        continue

    # 이니스프리 보고서에서 확인한 당기 금액은 네 번째 칸
    if row[0] == "매출액":
        revenue = parse_amount(row[3])

    elif row[0] in ("영업이익", "영업이익(손실)"):
        operating_profit = parse_amount(row[3])


if revenue is None or operating_profit is None:
    print("매출액 또는 영업이익을 찾지 못했습니다. 표 구조를 다시 확인해 주세요.")
else:
    financial_summary = pd.DataFrame([{
        "company_id": report["company_id"],
        "기업명": report["기업명"],
        "연도": 2025,
        "기간": "연간",
        "매출액": revenue,
        "영업이익": operating_profit,
        "단위": "원",
        "접수번호": report["접수번호"],
    }])

    display(financial_summary)

,company_id,기업명,연도,기간,매출액,영업이익,단위,접수번호
0,COMPANY-INNISFREE,이니스프리,2025,연간,209848553422,13348911731,원,20260401003769


In [74]:
# 이니스프리의 분기·반기 보고서 검색
response = requests.get(
    "https://opendart.fss.or.kr/api/list.json",
    params={
        "crtfc_key": dart_api_key,
        "corp_code": "00812353",  # 이니스프리
        "bgn_de": "20250101",
        "end_de": "20261231",
        "last_reprt_at": "Y",
        "page_count": 100,
    },
    timeout=30,
)

if response.status_code != 200:
    print(f"HTTP 오류: {response.status_code}")
else:
    data = response.json()

    if data.get("status") != "000":
        print(f'{data.get("status")} - {data.get("message")}')
    else:
        items = data.get("list", [])
        quarterly_reports = [
            {
                "보고서명": item.get("report_nm"),
                "접수일": item.get("rcept_dt"),
                "접수번호": item.get("rcept_no"),
            }
            for item in items
            if "분기보고서" in item.get("report_nm", "")
            or "반기보고서" in item.get("report_nm", "")
        ]

        if quarterly_reports:
            display(pd.DataFrame(quarterly_reports).head(10))
        else:
            print("첫 번째 조회 페이지에서 분기·반기 보고서를 찾지 못했습니다.")

첫 번째 조회 페이지에서 분기·반기 보고서를 찾지 못했습니다.


In [75]:
# 앞에서 찾은 감사보고서 중 두 번째 보고서
if not reports:
    print("조회된 감사보고서가 없습니다. 보고서 목록 조회 셀을 먼저 실행해 주세요.")
else:
    report = reports[1]

    print("기업명:", report["기업명"])
    print("보고서명:", report["보고서명"])
    print("접수번호:", report["접수번호"])

    # 접수번호로 공시 원문 요청
    response = requests.get(
        "https://opendart.fss.or.kr/api/document.xml",
        params={
            "crtfc_key": dart_api_key,
            "rcept_no": report["접수번호"],
        },
        timeout=60,
    )

    if response.status_code != 200:
        print("HTTP 오류:", response.status_code)

    elif not zipfile.is_zipfile(io.BytesIO(response.content)):
        # 정상적으로 ZIP이 오지 않았다면 Open DART 오류 메시지 확인
        try:
            error = ET.fromstring(response.content)
            print("조회 실패:", error.findtext("message"))
        except ET.ParseError:
            print("예상하지 못한 응답을 받았습니다.")

    else:
        with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
            file_names = archive.namelist()

            print("\nZIP 안의 파일 목록:")
            for name in file_names:
                print("-", name)

기업명: 에스쁘아
보고서명: 감사보고서 (2025.12)
접수번호: 20260326001214

ZIP 안의 파일 목록:
- 20260326001214_00760.xml


In [76]:
from html.parser import HTMLParser

# XML의 태그를 제외하고 글자만 모으는 도구
class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_data(self, data):
        text = data.strip()
        if text:
            self.parts.append(text)


# 앞 셀에서 받은 ZIP을 다시 열어 XML 읽기
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    xml_name = file_names[0]
    xml_content = archive.read(xml_name).decode("utf-8", errors="replace")

# XML에서 텍스트만 꺼내기
extractor = TextExtractor()
extractor.feed(xml_content)
report_text = " ".join(extractor.parts)

# 찾고 싶은 단어가 나오는 위치와 주변 내용 확인
for keyword in ["매출액", "영업이익"]:
    print(f"\n{'=' * 20} {keyword} {'=' * 20}")

    position = 0
    found = 0

    while found < 3:  # 단어별 최대 3곳 확인
        index = report_text.find(keyword, position)

        if index == -1:
            break

        start = max(0, index - 150)
        end = min(len(report_text), index + 350)

        print(f"\n[{found + 1}번째 발견]")
        print(report_text[start:end])

        position = index + len(keyword)
        found += 1

    if found == 0:
        print("해당 단어를 찾지 못했습니다.")


==================== 매출액 ====================

[1번째 발견]
괄 손 익 계 산 서 제 11 기 : 2025년 1월 1일부터 2025년 12월 31일까지 제 10 기 : 2024년 1월 1일부터 2024년 12월 31일까지 주식회사 에스쁘아 (단위 : 원) 과             목 주석 제 11 (당) 기 제 10 (전) 기 매출액 21 77,210,868,308 69,934,644,407 매출원가 22 34,284,372,809 29,542,701,874 매출총이익 42,926,495,499 40,391,942,533 판매비와관리비 22,23 43,379,120,365 37,598,804,046 영업이익 (452,624,866) 2,793,138,487 금융수익 4,24 142,224,252 183,740,250 금융비용 4,24 124,055,649 90,715,160 기타영업외수익(비용) 25 (80,978,895) 32,119,660 법인세비용차감전순이익 (515,435,158) 2,918,283,237 법인세비용(환입) 20 (2,

[2번째 발견]
2개월 이하인 리스이며, 소액리스자산은 소액의 사무실비품 등으로 구성되어 있습니다.① 변동리스료일부 부동산 리스는 매장에서 생기는 매출에 연동되는 변동리스료 조건을 포함합니다. 개별 매장의 경우, 리스료의 최대 100%까지 변동리스료 지급 조건을 따르며 리스료 산정시 매출액에 적용되는 비율 또한 그 범위가 넓습니다. 변동리스료를 지급하는 계약조건은 신규 매장의 고정원가 최소화를 포함하여 다양한 이유로 사용됩니다. 매출에 연동하는 변동리스료는 변동리스료를 유발하는 조건이 생기는 기간에 당기손익으로 인식합니다. ② 연장선택권 및 종료선택권 회사 전체에 걸쳐 다수의 부동산 및 시설장치 리스계약에 연장선택권 및 종료선택권을 포함하고 있습니다. 이러한 조건들은 계약 관리 측면에서 운영상의 유연성을 극대화하기 위해 사용됩니다. 보유하고 있는 대부분의 연장선택권 및 종료선택권

In [77]:
extractor = TableExtractor()
extractor.feed(xml_content)

# 매출액과 영업이익이 함께 있는 표 찾기
for table_number, table in enumerate(extractor.tables, start=1):
    table_text = " ".join(" ".join(row) for row in table)

    if "매출액" in table_text and "영업이익" in table_text:
        print(f"\n===== {table_number}번 표 =====")

        for row in table:
            print(row)

extractor = TableExtractor()
extractor.feed(xml_content)


===== 17번 표 =====
['과             목', '주석', '제 11 (당) 기', '제 10 (전) 기']
['매출액', '21', '', '77,210,868,308', '', '69,934,644,407']
['매출원가', '22', '', '34,284,372,809', '', '29,542,701,874']
['매출총이익', '', '', '42,926,495,499', '', '40,391,942,533']
['판매비와관리비', '22,23', '43,379,120,365', '', '37,598,804,046', '']
['영업이익', '', '', '(452,624,866)', '', '2,793,138,487']
['금융수익', '4,24', '142,224,252', '', '183,740,250', '']
['금융비용', '4,24', '124,055,649', '', '90,715,160', '']
['기타영업외수익(비용)', '25', '(80,978,895)', '', '32,119,660', '']
['법인세비용차감전순이익', '', '', '(515,435,158)', '', '2,918,283,237']
['법인세비용(환입)', '20', '(2,818,403)', '', '85,940,601', '']
['당기순이익(손실)', '', '', '(512,616,755)', '', '2,832,342,636']
['당기손익으로 재분류되지 않는 항목', '', '', '(65,521,654)', '', '(592,666,570)']
['순확정급여부채 재측정요소', '16, 20', '(65,521,654)', '', '(592,666,570)', '']
['당기총포괄이익(손실)', '', '', '(578,138,409)', '', '2,239,676,066']


In [78]:
# 화면에서 확인한 17번 표
income_table = extractor.tables[16]  # 파이썬에서는 첫 번째가 0번

# 괄호로 표시된 손실도 음수로 변환
def parse_amount(value):
    value = value.strip().replace(",", "")

    if value.startswith("(") and value.endswith(")"):
        return -int(value[1:-1])

    return int(value)


revenue = None
operating_profit = None

for row in income_table:
    if not row:
        continue

    # 이니스프리 보고서에서 확인한 당기 금액은 네 번째 칸
    if row[0] == "매출액":
        revenue = parse_amount(row[3])

    elif row[0] in ("영업이익", "영업이익(손실)"):
        operating_profit = parse_amount(row[3])


if revenue is None or operating_profit is None:
    print("매출액 또는 영업이익을 찾지 못했습니다. 표 구조를 다시 확인해 주세요.")
else:
    financial_summary = pd.DataFrame([{
        "company_id": report["company_id"],
        "기업명": report["기업명"],
        "연도": 2025,
        "기간": "연간",
        "매출액": revenue,
        "영업이익": operating_profit,
        "단위": "원",
        "접수번호": report["접수번호"],
    }])

    display(financial_summary)

,company_id,기업명,연도,기간,매출액,영업이익,단위,접수번호
0,COMPANY-ESPOIR,에스쁘아,2025,연간,77210868308,-452624866,원,20260326001214


In [81]:
# 모든 감사보고서에서 매출액·영업이익이 함께 있는 표 찾기
candidates = []
report_status = []

for report_item in reports:
    company_name = report_item["기업명"]
    receipt_no = report_item["접수번호"]

    try:
        response = requests.get(
            "https://opendart.fss.or.kr/api/document.xml",
            params={
                "crtfc_key": dart_api_key,
                "rcept_no": receipt_no,
            },
            timeout=60,
        )
    except requests.RequestException:
        report_status.append({
            "기업명": company_name,
            "결과": "요청 실패",
        })
        continue

    if response.status_code != 200:
        report_status.append({
            "기업명": company_name,
            "결과": f"HTTP 오류 {response.status_code}",
        })
        continue

    if not zipfile.is_zipfile(io.BytesIO(response.content)):
        report_status.append({
            "기업명": company_name,
            "결과": "보고서 원문을 가져오지 못함",
        })
        continue

    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        xml_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".xml")
        ]

        if not xml_files:
            report_status.append({
                "기업명": company_name,
                "결과": "XML 파일 없음",
            })
            continue

        xml_content = archive.read(xml_files[0]).decode(
            "utf-8", errors="replace"
        )

    # 보고서의 표들을 읽기
    table_extractor = TableExtractor()
    table_extractor.feed(xml_content)

    found = 0

    for table_number, table in enumerate(table_extractor.tables, start=1):
        revenue_rows = [
            row for row in table
            if row and row[0].strip() == "매출액"
        ]
        profit_rows = [
            row for row in table
            if row and row[0].strip() in ("영업이익", "영업이익(손실)")
        ]

        if revenue_rows and profit_rows:
            candidates.append({
                "company_id": report_item["company_id"],
                "기업명": company_name,
                "보고서명": report_item["보고서명"],
                "접수번호": receipt_no,
                "표 번호": table_number,
                "표 머리글": table[0],
                "매출액 행": revenue_rows[0],
                "영업이익 행": profit_rows[0],
            })
            found += 1

    report_status.append({
        "기업명": company_name,
        "결과": f"후보 표 {found}개" if found else "후보 표를 찾지 못함",
    })

print("기업별 조회 결과")
display(pd.DataFrame(report_status))

print("매출액·영업이익 표 후보")
candidates_df = pd.DataFrame(candidates)
display(candidates_df)

기업별 조회 결과


,기업명,결과
0,이니스프리,후보 표 1개
1,에스쁘아,후보 표 1개
2,에뛰드,후보 표 1개
3,아모스프로페셔널,후보 표 1개
4,코스알엑스,후보 표 1개


매출액·영업이익 표 후보


,company_id,기업명,보고서명,접수번호,표 번호,표 머리글,매출액 행,영업이익 행
0,COMPANY-INNISFREE,이니스프리,감사보고서 (2025.12),20260401003769,15,"[과 목, 주석, 제 17 (당) 기, 제 16(전) 기]","[매출액, 19,28, , 209,848,553,422, , 224,608,927,934]","[영업이익(손실), , , 13,348,911,731, , 1,635,369,529]"
1,COMPANY-ESPOIR,에스쁘아,감사보고서 (2025.12),20260326001214,17,"[과 목, 주석, 제 11 (당) 기, 제 10 (전) 기]","[매출액, 21, , 77,210,868,308, , 69,934,644,407]","[영업이익, , , (452,624,866), , 2,793,138,487]"
2,COMPANY-ETUDE,에뛰드,감사보고서 (2025.12),20260331004546,17,"[과목, 주석, 제 50 (당) 기말, 제 49 (전) 기말]","[매출액, 21,29, , 115,092,257,274, , 107,731,319,247]","[영업이익, , , 9,971,025,760, , 9,086,112,139]"
3,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,감사보고서 (2025.12),20260326000513,15,"[과 목, 주석, 제 50 (당) 기말, 제 49 (전) 기말]","[매출액, 19,27, , 79,136,115,848, , 79,234,373,217]","[영업이익, , , 6,982,117,086, , 11,070,105,450]"
4,COMPANY-COSRX,코스알엑스,감사보고서 (2025.12),20260402000034,17,"[과 목, 주석, 제13 (당) 기, 제12 (전) 기]","[매출액, 4,22, , 456,246,005,204, , 589,846,286,459]","[영업이익, 4, , 114,015,269,459, , 176,971,796,131]"


In [82]:
import re

def parse_candidate_amount(value):
    """쉼표와 괄호로 표시된 금액을 정수로 변환합니다."""
    value = str(value).strip().replace(",", "")

    if value.startswith("(") and value.endswith(")"):
        value = "-" + value[1:-1]

    if not re.fullmatch(r"-?\d+", value):
        return None

    return int(value)


rows = []

# 공통 기업 목록의 7개 기업을 기준으로 반복
for company in companies:
    company_id = company["company_id"]

    # 해당 기업의 후보 표 찾기
    if candidates_df.empty:
        company_candidates = pd.DataFrame()
    else:
        company_candidates = candidates_df[
            candidates_df["company_id"] == company_id
        ]

    result = {
        "company_id": company_id,
        "기업명": company["input_name"],
        "fiscal_year": 2025,
        "period": "연간",
        "revenue_krw": None,
        "operating_profit_krw": None,
        "statement_scope": None,
        "rcept_no": None,
        "확인 상태": "후보 보고서 또는 표 없음",
    }

    if len(company_candidates) == 1:
        candidate = company_candidates.iloc[0]
        revenue_row = candidate["매출액 행"]
        profit_row = candidate["영업이익 행"]

        result["rcept_no"] = candidate["접수번호"]

        # 이니스프리와 에스쁘아에서 확인한 구조:
        # 네 번째 칸에 당기 금액이 있음
        if company_id in ("COMPANY-INNISFREE", "COMPANY-ESPOIR"):
            if len(revenue_row) > 3 and len(profit_row) > 3:
                result["revenue_krw"] = parse_candidate_amount(revenue_row[3])
                result["operating_profit_krw"] = parse_candidate_amount(
                    profit_row[3]
                )

                if (
                    result["revenue_krw"] is not None
                    and result["operating_profit_krw"] is not None
                ):
                    result["확인 상태"] = "금액 확인 · 재무제표 범위 확인 필요"
                else:
                    result["확인 상태"] = "금액 확인 필요"

        else:
            result["확인 상태"] = "후보 표 발견 · 금액과 단위 확인 필요"

    elif len(company_candidates) > 1:
        result["확인 상태"] = "후보 표 여러 개 · 확인 필요"

    rows.append(result)

# 빈 값 때문에 금액이 소수로 바뀌지 않도록 정수형으로 지정
all_companies_df = pd.DataFrame(rows)

for column in ["revenue_krw", "operating_profit_krw"]:
    all_companies_df[column] = all_companies_df[column].astype("Int64")

display(all_companies_df)

,company_id,기업명,fiscal_year,period,revenue_krw,operating_profit_krw,statement_scope,rcept_no,확인 상태
0,COMPANY-INNISFREE,이니스프리,2025,연간,209848553422,13348911731,None,20260401003769,금액 확인 · 재무제표 범위 확인 필요
1,COMPANY-ESPOIR,에스쁘아,2025,연간,77210868308,-452624866,None,20260326001214,금액 확인 · 재무제표 범위 확인 필요
2,COMPANY-AMOREPACIFIC,아모레퍼시픽,2025,연간,<NA>,<NA>,None,NaN,후보 보고서 또는 표 없음
3,COMPANY-ETUDE,에뛰드,2025,연간,<NA>,<NA>,None,20260331004546,후보 표 발견 · 금액과 단위 확인 필요
4,COMPANY-PACIFICTECH,퍼시픽테크,2025,연간,<NA>,<NA>,None,NaN,후보 보고서 또는 표 없음
5,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,2025,연간,<NA>,<NA>,None,20260326000513,후보 표 발견 · 금액과 단위 확인 필요
6,COMPANY-COSRX,코스알엑스,2025,연간,<NA>,<NA>,None,20260402000034,후보 표 발견 · 금액과 단위 확인 필요


In [84]:
# company_info에 들어갈 재무 관련 스키마만 추출
company_info_finance_df = financial_candidates_df[[
    "company_id",
    "매출액 후보",
    "영업이익 후보",
]].rename(columns={
    "매출액 후보": "revenue_krw",
    "영업이익 후보": "operating_profit_krw",
}).copy()

company_info_finance_df["fiscal_year"] = 2025
company_info_finance_df["period"] = "연간"
company_info_finance_df["statement_scope"] = None

# 스키마 순서로 정리
company_info_finance_df = company_info_finance_df[[
    "company_id",
    "fiscal_year",
    "period",
    "revenue_krw",
    "operating_profit_krw",
    "statement_scope",
]]

display(company_info_finance_df)

,company_id,fiscal_year,period,revenue_krw,operating_profit_krw,statement_scope
0,COMPANY-INNISFREE,2025,연간,209848553422,13348911731,None
1,COMPANY-ESPOIR,2025,연간,77210868308,-452624866,None
2,COMPANY-ETUDE,2025,연간,115092257274,9971025760,None
3,COMPANY-AMOSPROFESSIONAL,2025,연간,79136115848,6982117086,None
4,COMPANY-COSRX,2025,연간,456246005204,114015269459,None


### 아모레퍼시픽의 경우 보고서가 없어 API를 통해 데이터 전처리

In [85]:
# 앞에서 재무제표 API로 조회한 아모레퍼시픽 데이터
amore = financial_df[
    financial_df["company_id"] == "COMPANY-AMOREPACIFIC"
]

# 매출액과 영업이익 행 찾기
revenue_rows = amore[amore["항목"] == "매출액"]
profit_rows = amore[amore["항목"] == "영업이익"]

if len(revenue_rows) != 1 or len(profit_rows) != 1:
    print("매출액 또는 영업이익 항목을 하나로 특정할 수 없습니다.")
    display(amore)
else:
    # API 금액의 쉼표를 제거하고 숫자로 변환
    revenue = int(str(revenue_rows.iloc[0]["당기 금액"]).replace(",", ""))
    operating_profit = int(
        str(profit_rows.iloc[0]["당기 금액"]).replace(",", "")
    )

    amore_row = pd.DataFrame([{
        "company_id": "COMPANY-AMOREPACIFIC",
        "fiscal_year": 2025,
        "period": "연간",
        "revenue_krw": revenue,
        "operating_profit_krw": operating_profit,
        "statement_scope": "개별",
    }])

    # 셀을 다시 실행해도 아모레퍼시픽이 중복되지 않도록 처리
    company_info_finance_df = company_info_finance_df[
        company_info_finance_df["company_id"] != "COMPANY-AMOREPACIFIC"
    ]

    company_info_finance_df = pd.concat(
        [company_info_finance_df, amore_row],
        ignore_index=True,
    )

    display(company_info_finance_df)

,company_id,fiscal_year,period,revenue_krw,operating_profit_krw,statement_scope
0,COMPANY-INNISFREE,2025,연간,209848553422,13348911731,None
1,COMPANY-ESPOIR,2025,연간,77210868308,-452624866,None
2,COMPANY-ETUDE,2025,연간,115092257274,9971025760,None
3,COMPANY-AMOSPROFESSIONAL,2025,연간,79136115848,6982117086,None
4,COMPANY-COSRX,2025,연간,456246005204,114015269459,None
5,COMPANY-AMOREPACIFIC,2025,연간,2653725793231,212514854678,개별


### 퍼시픽테크 데이터 조회 및 전처리

In [86]:
# 퍼시픽테크의 공시 목록 확인
url = "https://opendart.fss.or.kr/api/list.json"

pacific_reports = []
page = 1

while True:
    response = requests.get(
        url,
        params={
            "crtfc_key": dart_api_key,
            "corp_code": "01784251",  # 퍼시픽테크 DART 고유번호
            "bgn_de": "20250101",
            "end_de": "20261104",
            "last_reprt_at": "Y",
            "page_count": 100,
            "page_no": page,
        },
        timeout=30,
    )

    if response.status_code != 200:
        print(f"HTTP 오류: {response.status_code}")
        break

    data = response.json()

    if data.get("status") == "013":
        print("이 기간에 조회된 공시가 없습니다.")
        break

    if data.get("status") != "000":
        print(f'{data.get("status")} - {data.get("message")}')
        break

    for item in data.get("list", []):
        pacific_reports.append({
            "보고서명": item.get("report_nm"),
            "접수일": item.get("rcept_dt"),
            "접수번호": item.get("rcept_no"),
        })

    if page >= int(data.get("total_page", 1)):
        break

    page += 1

print(f"찾은 공시: {len(pacific_reports)}건")
display(pd.DataFrame(pacific_reports).head(10))

찾은 공시: 11건


,보고서명,접수일,접수번호
0,대규모기업집단현황공시[분기별공시(개별회사용)],20260828,20260828000811
1,지급수단별ㆍ지급기간별지급금액및분쟁조정기구에관한사항,20260813,20260813001522
2,대규모기업집단현황공시[연1회공시및1/4분기용(개별회사)],20260601,20260601000936
3,대규모기업집단현황공시[분기별공시(개별회사용)],20260227,20260227005822
4,지급수단별ㆍ지급기간별지급금액및분쟁조정기구에관한사항,20260213,20260213002207
5,대규모기업집단현황공시[분기별공시(개별회사용)],20251127,20251127001062
6,대규모기업집단현황공시[분기별공시(개별회사용)],20250829,20250829000655
7,지급수단별ㆍ지급기간별지급금액및분쟁조정기구에관한사항,20250814,20250814002151
8,대규모기업집단현황공시[연1회공시및1/4분기용(개별회사)],20250602,20250602000345
9,대규모기업집단현황공시[분기별공시(개별회사용)],20250228,20250228001274


In [87]:
# 퍼시픽테크 연 1회 공시 원문 가져오기
receipt_no = "20260601000936"

response = requests.get(
    "https://opendart.fss.or.kr/api/document.xml",
    params={
        "crtfc_key": dart_api_key,
        "rcept_no": receipt_no,
    },
    timeout=60,
)

if response.status_code != 200:
    print(f"HTTP 오류: {response.status_code}")

elif not zipfile.is_zipfile(io.BytesIO(response.content)):
    print("보고서 원문을 가져오지 못했습니다.")

else:
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        xml_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".xml")
        ]

        for xml_name in xml_files:
            xml_content = archive.read(xml_name).decode(
                "utf-8", errors="replace"
            )

            # 앞에서 만든 TextExtractor를 사용해 글자만 추출
            text_extractor = TextExtractor()
            text_extractor.feed(xml_content)
            report_text = " ".join(text_extractor.parts)

            print(f"\n파일: {xml_name}")

            for keyword in ["매출액", "영업이익"]:
                print(f"\n--- {keyword} ---")

                index = report_text.find(keyword)

                if index == -1:
                    print("찾지 못했습니다.")
                else:
                    start = max(0, index - 200)
                    end = min(len(report_text), index + 500)
                    print(report_text[start:end])


파일: 20260601000936.xml

--- 매출액 ---
성자산 비유동자산(b) 자산총계(a+b) 유동부채(c) 비유동부채(d) 부채총계(c+d) 부채총계 중 차입금 자본금 자본총계 비금융회사 (주)퍼시픽테크 1,522 928 604 2,126 1,035 17 1,051 - 2,500 1,074 97.86 (3) 회사 손익현황 : 개별 손익계산서 기준 손익현황 (직전 사업연도말 기준,단위 : 백만원) 소속회사명 매출액(또는 영업수익) 영업이익 기타수익등 기타비용등 기타비용등 중이자비용 당기순이익 비금융회사 (주)퍼시픽테크 1,854 -922 29 2 2 -895 (4) 국외 계열회사 현황 : 국내 계열회사가 최다출자자인 국외 계열회사의 현황 : 해당사항 없음 (5) 계열회사 변동 현황 : 국내ㆍ외 계열회사의 계열편입과 계열제외 내역 : 해당사항 없음 2. 임원 , 이사회 등의 운영현황[ESG] (1) 임원현황[ESG] : 회사의 등기임원 현황 (금년도 지정일 기준) 소속회사명 임원명 직위 취임일 등기일 임기 만료(예정)일 동일인과친족관계 겸직사항 주요경력 퇴직임직원 중임여부 비금융회사 (주)퍼시픽테크 주효정 대표이사(이사회 의장) 2024.07.15 2024.07.23 2027.07.15 - - 현) (주)아모레퍼시픽 디지털전략Division장 - 신임 주효정 사내이사 2024.07.15 2024.07.23 2027.07.15 - - 현) (주)아모레퍼시픽 디지털전략Division장 - 신

--- 영업이익 ---
자산총계(a+b) 유동부채(c) 비유동부채(d) 부채총계(c+d) 부채총계 중 차입금 자본금 자본총계 비금융회사 (주)퍼시픽테크 1,522 928 604 2,126 1,035 17 1,051 - 2,500 1,074 97.86 (3) 회사 손익현황 : 개별 손익계산서 기준 손익현황 (직전 사업연도말 기준,단위 : 백만원) 소속회사명 매출액(또는 영업수익) 영업이익 기타수익등 기타비용등 기타비용등 중이자비용 당기순이익 비금융회사 (주)퍼시픽테

In [88]:
result_df.loc[
    result_df["company_id"] == "COMPANY-PACIFICTECH",
    ["input_name", "acc_mt"]
]

,input_name,acc_mt
4,퍼시픽테크,12


In [89]:
# 퍼시픽테크 2025년 연간 손익현황
pacific_row = pd.DataFrame([{
    "company_id": "COMPANY-PACIFICTECH",
    "fiscal_year": 2025,
    "period": "연간",
    "revenue_krw": 1_854 * 1_000_000,
    "operating_profit_krw": -922 * 1_000_000,
    "statement_scope": "개별",
}])

# 셀을 다시 실행해도 중복되지 않도록 기존 행 제거
company_info_finance_df = company_info_finance_df[
    company_info_finance_df["company_id"] != "COMPANY-PACIFICTECH"
]

# 기존 표에 추가
company_info_finance_df = pd.concat(
    [company_info_finance_df, pacific_row],
    ignore_index=True,
)

display(company_info_finance_df)

,company_id,fiscal_year,period,revenue_krw,operating_profit_krw,statement_scope
0,COMPANY-INNISFREE,2025,연간,209848553422,13348911731,None
1,COMPANY-ESPOIR,2025,연간,77210868308,-452624866,None
2,COMPANY-ETUDE,2025,연간,115092257274,9971025760,None
3,COMPANY-AMOSPROFESSIONAL,2025,연간,79136115848,6982117086,None
4,COMPANY-COSRX,2025,연간,456246005204,114015269459,None
5,COMPANY-AMOREPACIFIC,2025,연간,2653725793231,212514854678,개별
6,COMPANY-PACIFICTECH,2025,연간,1854000000,-922000000,개별


### 개황데이터와 재무 데이터 합치기

In [90]:
# 기업개황과 재무 정보를 company_id 기준으로 결합
company_info_preview_df = overview_df.merge(
    company_info_finance_df,
    on="company_id",
    how="left",
    validate="one_to_one",  # 기업 ID가 중복되면 오류로 알려줌
)

# 아직 확인하지 않은 스키마 항목은 빈 값으로 둡니다.
company_info_preview_df["main_business"] = None
company_info_preview_df["employee_count"] = None

# company_info의 기업 기본 정보·재무 스키마 열
company_info_preview_df = company_info_preview_df[[
    "company_id",
    "industry",
    "main_business",
    "revenue_krw",
    "operating_profit_krw",
    "listing_status",
    "employee_count",
    "established_date",
    "fiscal_year",
    "period",
    "statement_scope",
]]

display(company_info_preview_df)

,company_id,industry,main_business,revenue_krw,operating_profit_krw,listing_status,employee_count,established_date,fiscal_year,period,statement_scope
0,COMPANY-INNISFREE,None,None,209848553422,13348911731,NaN,None,2009-11-10,2025,연간,None
1,COMPANY-ESPOIR,None,None,77210868308,-452624866,NaN,None,2015-01-02,2025,연간,None
2,COMPANY-AMOREPACIFIC,None,None,2653725793231,212514854678,상장,None,2006-06-01,2025,연간,개별
3,COMPANY-ETUDE,None,None,115092257274,9971025760,NaN,None,1966-02-19,2025,연간,None
4,COMPANY-PACIFICTECH,None,None,1854000000,-922000000,NaN,None,2023-07-27,2025,연간,개별
5,COMPANY-AMOSPROFESSIONAL,None,None,79136115848,6982117086,NaN,None,1976-01-24,2025,연간,None
6,COMPANY-COSRX,None,None,456246005204,114015269459,NaN,None,2013-12-11,2025,연간,None


In [91]:
# company_info에 필요한 전체 스키마
schema_columns = [
    # 공통 필드
    "record_id",
    "company_id",
    "document_id",
    "domain",
    "content",
    "evidence_text",
    "evidence_location",
    "as_of_date",
    "verification_status",

    # 기업 기본 정보·재무 필드
    "industry",
    "main_business",
    "revenue_krw",
    "operating_profit_krw",
    "listing_status",
    "employee_count",
    "established_date",
    "fiscal_year",
    "period",
    "statement_scope",
]

required_columns = {
    "record_id",
    "company_id",
    "document_id",
    "domain",
    "content",
    "verification_status",
}

# 스키마와 현재 표 비교
check_rows = []

for column in schema_columns:
    exists = column in company_info_preview_df.columns

    if exists:
        values = company_info_preview_df[column]
        # None, NaN, 빈 문자열을 모두 빈 값으로 판단
        missing = values.isna() | values.astype("string").str.strip().eq("").fillna(False)
        missing_count = int(missing.sum())
    else:
        missing_count = len(company_info_preview_df)

    check_rows.append({
        "필드명": column,
        "필수 여부": "필수" if column in required_columns else "확인 시",
        "열 존재": "있음" if exists else "없음",
        "빈 값 수": missing_count,
    })

schema_check_df = pd.DataFrame(check_rows)
display(schema_check_df)

,필드명,필수 여부,열 존재,빈 값 수
0,record_id,필수,없음,7
1,company_id,필수,있음,0
2,document_id,필수,없음,7
3,domain,필수,없음,7
4,content,필수,없음,7
5,evidence_text,확인 시,없음,7
6,evidence_location,확인 시,없음,7
7,as_of_date,확인 시,없음,7
8,verification_status,필수,없음,7
9,industry,확인 시,있음,7


In [92]:
from uuid import uuid4

# 이미 record_id가 있으면 유지하고, 없을 때만 생성
if "record_id" not in company_info_preview_df.columns:
    company_info_preview_df["record_id"] = [
        f"COMPANY_INFO-AN-{uuid4().hex}"
        for _ in range(len(company_info_preview_df))
    ]

# ID가 중복되지 않았는지 확인
print("record_id 중복 수:", company_info_preview_df["record_id"].duplicated().sum())

display(company_info_preview_df[["record_id", "company_id"]])

record_id 중복 수: 0


,record_id,company_id
0,COMPANY_INFO-AN-51db87bce62b4c7a8a8d032b6603341d,COMPANY-INNISFREE
1,COMPANY_INFO-AN-5fe432a786e041968ed5e70389de4b43,COMPANY-ESPOIR
2,COMPANY_INFO-AN-bd33a90f54084386bb0c614d4380366c,COMPANY-AMOREPACIFIC
3,COMPANY_INFO-AN-967b9ffe3f9f4d4698fd5d89fab4b19a,COMPANY-ETUDE
4,COMPANY_INFO-AN-72a39747e28748899b4026315279dff2,COMPANY-PACIFICTECH
5,COMPANY_INFO-AN-75336c85fe8a4640b2a440f09a7d988b,COMPANY-AMOSPROFESSIONAL
6,COMPANY_INFO-AN-f2fe08b88b13468ca1ab6d29e746d69c,COMPANY-COSRX


In [93]:
# company_info에 필요한 전체 스키마
schema_columns = [
    # 공통 필드
    "record_id",
    "company_id",
    "document_id",
    "domain",
    "content",
    "evidence_text",
    "evidence_location",
    "as_of_date",
    "verification_status",

    # 기업 기본 정보·재무 필드
    "industry",
    "main_business",
    "revenue_krw",
    "operating_profit_krw",
    "listing_status",
    "employee_count",
    "established_date",
    "fiscal_year",
    "period",
    "statement_scope",
]

required_columns = {
    "record_id",
    "company_id",
    "document_id",
    "domain",
    "content",
    "verification_status",
}

# 스키마와 현재 표 비교
check_rows = []

for column in schema_columns:
    exists = column in company_info_preview_df.columns

    if exists:
        values = company_info_preview_df[column]
        # None, NaN, 빈 문자열을 모두 빈 값으로 판단
        missing = values.isna() | values.astype("string").str.strip().eq("").fillna(False)
        missing_count = int(missing.sum())
    else:
        missing_count = len(company_info_preview_df)

    check_rows.append({
        "필드명": column,
        "필수 여부": "필수" if column in required_columns else "확인 시",
        "열 존재": "있음" if exists else "없음",
        "빈 값 수": missing_count,
    })

schema_check_df = pd.DataFrame(check_rows)
display(schema_check_df)

,필드명,필수 여부,열 존재,빈 값 수
0,record_id,필수,있음,0
1,company_id,필수,있음,0
2,document_id,필수,없음,7
3,domain,필수,없음,7
4,content,필수,없음,7
5,evidence_text,확인 시,없음,7
6,evidence_location,확인 시,없음,7
7,as_of_date,확인 시,없음,7
8,verification_status,필수,없음,7
9,industry,확인 시,있음,7


In [94]:
import hashlib

def make_document_id(company_id):
    # 기업 ID를 바탕으로 항상 같은 문서 ID 생성
    digest = hashlib.sha256(company_id.encode("utf-8")).hexdigest()[:16]
    return f"DOC-AN-{digest}"

# 기업별로 document_id 하나씩 추가
company_info_preview_df["document_id"] = (
    company_info_preview_df["company_id"].apply(make_document_id)
)

display(
    company_info_preview_df[[
        "company_id",
        "document_id",
    ]]
)

,company_id,document_id
0,COMPANY-INNISFREE,DOC-AN-18d7d1d2b297f978
1,COMPANY-ESPOIR,DOC-AN-52452357113572f8
2,COMPANY-AMOREPACIFIC,DOC-AN-ddcfaa91e401c8f3
3,COMPANY-ETUDE,DOC-AN-dd2f03c6d23a0656
4,COMPANY-PACIFICTECH,DOC-AN-7d1317b01f768312
5,COMPANY-AMOSPROFESSIONAL,DOC-AN-64e23787de466c90
6,COMPANY-COSRX,DOC-AN-c6cb42492e98cd6c


In [95]:
# company_info에 필요한 전체 스키마
schema_columns = [
    # 공통 필드
    "record_id",
    "company_id",
    "document_id",
    "domain",
    "content",
    "evidence_text",
    "evidence_location",
    "as_of_date",
    "verification_status",

    # 기업 기본 정보·재무 필드
    "industry",
    "main_business",
    "revenue_krw",
    "operating_profit_krw",
    "listing_status",
    "employee_count",
    "established_date",
    "fiscal_year",
    "period",
    "statement_scope",
]

required_columns = {
    "record_id",
    "company_id",
    "document_id",
    "domain",
    "content",
    "verification_status",
}

# 스키마와 현재 표 비교
check_rows = []

for column in schema_columns:
    exists = column in company_info_preview_df.columns

    if exists:
        values = company_info_preview_df[column]
        # None, NaN, 빈 문자열을 모두 빈 값으로 판단
        missing = values.isna() | values.astype("string").str.strip().eq("").fillna(False)
        missing_count = int(missing.sum())
    else:
        missing_count = len(company_info_preview_df)

    check_rows.append({
        "필드명": column,
        "필수 여부": "필수" if column in required_columns else "확인 시",
        "열 존재": "있음" if exists else "없음",
        "빈 값 수": missing_count,
    })

schema_check_df = pd.DataFrame(check_rows)
display(schema_check_df)

,필드명,필수 여부,열 존재,빈 값 수
0,record_id,필수,있음,0
1,company_id,필수,있음,0
2,document_id,필수,있음,0
3,domain,필수,없음,7
4,content,필수,없음,7
5,evidence_text,확인 시,없음,7
6,evidence_location,확인 시,없음,7
7,as_of_date,확인 시,없음,7
8,verification_status,필수,없음,7
9,industry,확인 시,있음,7


In [ ]:
# company_info의 전체 스키마 순서
schema_columns = [
    "record_id",
    "company_id",
    "document_id",
    "domain",
    "content",
    "evidence_text",
    "evidence_location",
    "as_of_date",
    "verification_status",
    "industry",
    "main_business",
    "revenue_krw",
    "operating_profit_krw",
    "listing_status",
    "employee_count",
    "established_date",
    "fiscal_year",
    "period",
    "statement_scope",
]

# 없는 열은 빈 값으로 표시
company_info_schema_df = company_info_preview_df.reindex(
    columns=schema_columns
)

# 모든 열과 전체 문자열 표시
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

display(company_info_schema_df)

,record_id,company_id,document_id,domain,content,evidence_text,evidence_location,as_of_date,verification_status,industry,main_business,revenue_krw,operating_profit_krw,listing_status,employee_count,established_date,fiscal_year,period,statement_scope
0,COMPANY_INFO-AN-51db87bce62b4c7a8a8d032b6603341d,COMPANY-INNISFREE,DOC-AN-18d7d1d2b297f978,NaN,NaN,NaN,NaN,NaN,NaN,None,None,209848553422,13348911731,NaN,None,2009-11-10,2025,연간,None
1,COMPANY_INFO-AN-5fe432a786e041968ed5e70389de4b43,COMPANY-ESPOIR,DOC-AN-52452357113572f8,NaN,NaN,NaN,NaN,NaN,NaN,None,None,77210868308,-452624866,NaN,None,2015-01-02,2025,연간,None
2,COMPANY_INFO-AN-bd33a90f54084386bb0c614d4380366c,COMPANY-AMOREPACIFIC,DOC-AN-ddcfaa91e401c8f3,NaN,NaN,NaN,NaN,NaN,NaN,None,None,2653725793231,212514854678,상장,None,2006-06-01,2025,연간,개별
3,COMPANY_INFO-AN-967b9ffe3f9f4d4698fd5d89fab4b19a,COMPANY-ETUDE,DOC-AN-dd2f03c6d23a0656,NaN,NaN,NaN,NaN,NaN,NaN,None,None,115092257274,9971025760,NaN,None,1966-02-19,2025,연간,None
4,COMPANY_INFO-AN-72a39747e28748899b4026315279dff2,COMPANY-PACIFICTECH,DOC-AN-7d1317b01f768312,NaN,NaN,NaN,NaN,NaN,NaN,None,None,1854000000,-922000000,NaN,None,2023-07-27,2025,연간,개별
5,COMPANY_INFO-AN-75336c85fe8a4640b2a440f09a7d988b,COMPANY-AMOSPROFESSIONAL,DOC-AN-64e23787de466c90,NaN,NaN,NaN,NaN,NaN,NaN,None,None,79136115848,6982117086,NaN,None,1976-01-24,2025,연간,None
6,COMPANY_INFO-AN-f2fe08b88b13468ca1ab6d29e746d69c,COMPANY-COSRX,DOC-AN-c6cb42492e98cd6c,NaN,NaN,NaN,NaN,NaN,NaN,None,None,456246005204,114015269459,NaN,None,2013-12-11,2025,연간,None


In [97]:
company_info_schema_df.shape

(7, 19)

In [98]:
company_info_schema_df.columns

Index(['record_id', 'company_id', 'document_id', 'domain', 'content',
       'evidence_text', 'evidence_location', 'as_of_date',
       'verification_status', 'industry', 'main_business', 'revenue_krw',
       'operating_profit_krw', 'listing_status', 'employee_count',
       'established_date', 'fiscal_year', 'period', 'statement_scope'],
      dtype='str')

In [100]:
import pandas as pd

# 기업개황 API 문서 주소입니다.
# 인증키가 들어간 실제 요청 URL은 저장하지 않습니다.
COMPANY_API_URL = "https://opendart.fss.or.kr/api/company.json"


def format_dart_date(value):
    """DART 날짜 YYYYMMDD를 YYYY-MM-DD로 바꿉니다."""

    # 값이 없으면 None으로 둡니다.
    if value is None or pd.isna(value) or not str(value).strip():
        return None

    # 날짜 형식이 맞지 않으면 오류 대신 NaT가 됩니다.
    date = pd.to_datetime(
        str(value).strip(),
        format="%Y%m%d",
        errors="coerce",
    )

    if pd.isna(date):
        return None

    return date.strftime("%Y-%m-%d")


def convert_listing_status(corp_cls):
    """DART 법인구분에서 확인 가능한 상장 상태만 변환합니다."""

    # Y: 유가증권시장, K: 코스닥, N: 코넥스
    if corp_cls in ("Y", "K", "N"):
        return "상장"

    # E는 '기타'이므로 비상장이라고 단정하지 않습니다.
    return None


def make_api_source(field_name):
    """기업개황 API의 특정 필드를 확인한 출처로 기록합니다."""

    return {
        "source_type": "public_data",
        "source_name": "Open DART 기업개황 API",
        "title": "기업개황 조회",
        "original_url": COMPANY_API_URL,
        "published_at": None,

        # 실제 API 호출 시각을 별도로 기록하지 않았으므로 비워 둡니다.
        "collected_at": None,

        # API 응답에 정보의 기준일이 없으면 임의로 넣지 않습니다.
        "as_of_date": None,

        # API 응답에서 어느 필드를 확인했는지 기록합니다.
        "evidence_location": field_name,

        "raw_path": None,
    }


# API 결과에서 같은 기업 ID가 중복되면 먼저 확인합니다.
if not result_df.empty and result_df["company_id"].duplicated().any():
    raise ValueError(
        "기업개황 결과에 중복된 company_id가 있습니다. 먼저 확인해 주세요."
    )

# 기업 ID를 기준으로 API 결과를 쉽게 찾을 수 있도록 만듭니다.
# 예: api_by_company["COMPANY-ETUDE"]
api_by_company = {}

if not result_df.empty:
    for _, row in result_df.iterrows():
        api_by_company[row["company_id"]] = row


# 새 company_info 데이터를 저장할 리스트입니다.
company_info_records = []

# 공통 기업 목록의 7개 기업을 하나씩 처리합니다.
for company in companies:
    company_id = company["company_id"]

    # API 조회에 실패한 기업이면 None이 됩니다.
    api_row = api_by_company.get(company_id)

    # 먼저 모든 값을 확인되지 않은 상태로 만듭니다.
    record = {
        "company_id": company_id,
        "industry": None,
        "main_business": None,
        "listing_status": None,
        "employee_count": None,
        "established_date": None,
        "field_sources": {},
    }

    # 기업개황 조회에 성공한 경우에만 API 값을 채웁니다.
    if api_row is not None:

        # 1. 법인 설립일
        established_date = format_dart_date(api_row.get("est_dt"))

        if established_date is not None:
            record["established_date"] = established_date

            # established_date의 출처는 API의 est_dt 필드입니다.
            record["field_sources"]["established_date"] = [
                make_api_source("est_dt")
            ]

        # 2. 상장 상태
        listing_status = convert_listing_status(api_row.get("corp_cls"))

        if listing_status is not None:
            record["listing_status"] = listing_status

            # listing_status의 출처는 API의 corp_cls 필드입니다.
            record["field_sources"]["listing_status"] = [
                make_api_source("corp_cls")
            ]

        # industry는 API의 induty_code가 업종 '코드'이므로
        # 분류명을 확인하기 전까지 None으로 둡니다.
        #
        # 주요 사업과 직원 수도 이 API 응답만으로는 채우지 않습니다.

    # 완성한 기업 기본정보를 리스트에 추가합니다.
    company_info_records.append(record)


print(f"기업 기본정보 구성: {len(company_info_records)}개")

기업 기본정보 구성: 7개


In [101]:
# JSON 객체 목록을 표로 변환합니다.
company_info_df = pd.DataFrame(company_info_records)

# 모든 열을 볼 수 있게 설정합니다.
pd.set_option("display.max_columns", None)

# 표에서 앞의 10개를 보여줍니다.
display(company_info_df.head(10))

,company_id,industry,main_business,listing_status,employee_count,established_date,field_sources
0,COMPANY-INNISFREE,None,None,NaN,None,2009-11-10,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
1,COMPANY-ESPOIR,None,None,NaN,None,2015-01-02,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
2,COMPANY-AMOREPACIFIC,None,None,상장,None,2006-06-01,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}], 'listing_status': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'corp_cls', 'raw_path': None}]}"
3,COMPANY-ETUDE,None,None,NaN,None,1966-02-19,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
4,COMPANY-PACIFICTECH,None,None,NaN,None,2023-07-27,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
5,COMPANY-AMOSPROFESSIONAL,None,None,NaN,None,1976-01-24,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
6,COMPANY-COSRX,None,None,NaN,None,2013-12-11,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"


In [106]:
# 공통 기업 목록에서 확인에 필요한 컬럼을 선택합니다.
company_list = companies_df[[
    "company_id",
    "input_name",
    "dart_corp_code",
]].copy()

# API 조회 결과가 비어 있는지 확인합니다.
if result_df.empty:
    print("기업개황 조회 결과가 없습니다.")
    print("앞의 기업개황 API 조회 셀을 먼저 실행해 주세요.")

else:
    # API에서 필요한 컬럼만 가져옵니다.
    # induty_code: 업종코드
    # corp_cls: 법인구분
    # est_dt: 설립일
    api_info = result_df.reindex(columns=[
        "company_id",
        "induty_code",
        "corp_cls",
        "est_dt",
    ]).copy()

    # 공통 기업 목록과 API 결과를 기업 ID로 연결합니다.
    # how="left"를 사용하면 API 조회에 실패한 기업도 목록에 남습니다.
    basic_check_df = company_list.merge(
        api_info,
        on="company_id",
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    # 표에서 보기 쉽게 컬럼명을 한글로 바꿉니다.
    basic_check_df = basic_check_df.rename(columns={
        "company_id": "기업 ID",
        "input_name": "기업명",
        "dart_corp_code": "DART 고유번호",
        "induty_code": "업종코드",
        "corp_cls": "법인구분",
        "est_dt": "설립일 원본",
        "_merge": "API 조회",
    })

    # API 결과가 연결됐는지 한글로 표시합니다.
    basic_check_df["API 조회"] = basic_check_df["API 조회"].map({
        "both": "성공",
        "left_only": "결과 없음",
        "right_only": "목록 밖 결과",
    })

    # 우리 기업은 7개라서 모두 표시됩니다.
    display(basic_check_df.head(10))

,기업 ID,기업명,DART 고유번호,업종코드,법인구분,설립일 원본,API 조회
0,COMPANY-INNISFREE,이니스프리,00812353,46443,E,20091110,성공
1,COMPANY-ESPOIR,에스쁘아,01061150,46443,E,20150102,성공
2,COMPANY-AMOREPACIFIC,아모레퍼시픽,00583424,20423,Y,20060601,성공
3,COMPANY-ETUDE,에뛰드,00183808,46443,E,19660219,성공
4,COMPANY-PACIFICTECH,퍼시픽테크,01784251,28519,E,20230727,성공
5,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,00144456,46443,E,19760124,성공
6,COMPANY-COSRX,코스알엑스,01364491,20423,E,20131211,성공


In [107]:
# 확인한 업종코드와 업종명을 연결합니다.
# 코드는 앞에 0이 들어갈 가능성도 있으므로 문자열로 관리합니다.
industry_names = {
    "46443": "화장품 및 화장용품 도매업",
    "20423": "화장품 제조업",
    "28519": "기타 가정용 전기 기기 제조업",
}

# 원래 표를 그대로 두고, 확인용 표를 복사합니다.
industry_check_df = basic_check_df.copy()

# 업종코드를 문자열로 바꾸고 양쪽 공백을 제거합니다.
industry_check_df["업종코드"] = (
    industry_check_df["업종코드"]
    .astype("string")
    .str.strip()
)

# 업종코드에 해당하는 업종명을 새 컬럼에 넣습니다.
# 위 딕셔너리에 없는 코드는 빈 값으로 표시됩니다.
industry_check_df["업종명"] = (
    industry_check_df["업종코드"]
    .map(industry_names)
)

# 기업명과 업종코드·업종명을 확인합니다.
display(
    industry_check_df[
        ["기업명", "업종코드", "업종명"]
    ]
)

,기업명,업종코드,업종명
0,이니스프리,46443,화장품 및 화장용품 도매업
1,에스쁘아,46443,화장품 및 화장용품 도매업
2,아모레퍼시픽,20423,화장품 제조업
3,에뛰드,46443,화장품 및 화장용품 도매업
4,퍼시픽테크,28519,기타 가정용 전기 기기 제조업
5,아모스프로페셔널,46443,화장품 및 화장용품 도매업
6,코스알엑스,20423,화장품 제조업


In [108]:
# 확인한 업종코드와 업종명
industry_names = {
    "46443": "화장품 및 화장용품 도매업",
    "20423": "화장품 제조업",
    "28519": "기타 가정용 전기 기기 제조업",
}

# 기업개황에서 기업 ID와 업종코드를 가져오기
industry_by_company = (
    overview_df
    .set_index("company_id")["industry_code"]
    .astype("string")
    .str.strip()
    .map(industry_names)
)

# 기존 미리보기의 industry 열 채우기
company_info_preview_df["industry"] = (
    company_info_preview_df["company_id"]
    .map(industry_by_company)
    .combine_first(company_info_preview_df["industry"])
)

# 기업별 반영 결과 확인
display(company_info_preview_df[["company_id", "industry"]])

,company_id,industry
0,COMPANY-INNISFREE,화장품 및 화장용품 도매업
1,COMPANY-ESPOIR,화장품 및 화장용품 도매업
2,COMPANY-AMOREPACIFIC,화장품 제조업
3,COMPANY-ETUDE,화장품 및 화장용품 도매업
4,COMPANY-PACIFICTECH,기타 가정용 전기 기기 제조업
5,COMPANY-AMOSPROFESSIONAL,화장품 및 화장용품 도매업
6,COMPANY-COSRX,화장품 제조업


In [109]:
display(company_info_df.head(10))

,company_id,industry,main_business,listing_status,employee_count,established_date,field_sources
0,COMPANY-INNISFREE,None,None,NaN,None,2009-11-10,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
1,COMPANY-ESPOIR,None,None,NaN,None,2015-01-02,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
2,COMPANY-AMOREPACIFIC,None,None,상장,None,2006-06-01,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}], 'listing_status': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'corp_cls', 'raw_path': None}]}"
3,COMPANY-ETUDE,None,None,NaN,None,1966-02-19,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
4,COMPANY-PACIFICTECH,None,None,NaN,None,2023-07-27,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
5,COMPANY-AMOSPROFESSIONAL,None,None,NaN,None,1976-01-24,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
6,COMPANY-COSRX,None,None,NaN,None,2013-12-11,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"


In [110]:
# 확인한 업종코드와 업종명
industry_names = {
    "46443": "화장품 및 화장용품 도매업",
    "20423": "화장품 제조업",
    "28519": "기타 가정용 전기 기기 제조업",
}

# 기업개황 표에 업종명 반영
overview_df["industry"] = (
    overview_df["industry_code"]
    .astype("string")
    .str.strip()
    .map(industry_names)
)

# company_id별 업종명
industry_by_company = overview_df.set_index("company_id")["industry"]

# 이미 만든 결합 표가 있다면 그 표에도 반영
if "company_info_preview_df" in globals():
    company_info_preview_df["industry"] = (
        company_info_preview_df["company_id"].map(industry_by_company)
    )

# 이미 만든 스키마 표가 있다면 그 표에도 반영
if "company_info_schema_df" in globals():
    company_info_schema_df["industry"] = (
        company_info_schema_df["company_id"].map(industry_by_company)
    )

# 기업개황 표에서 확인
display(overview_df[["company_id", "industry_code", "industry"]])

,company_id,industry_code,industry
0,COMPANY-INNISFREE,46443,화장품 및 화장용품 도매업
1,COMPANY-ESPOIR,46443,화장품 및 화장용품 도매업
2,COMPANY-AMOREPACIFIC,20423,화장품 제조업
3,COMPANY-ETUDE,46443,화장품 및 화장용품 도매업
4,COMPANY-PACIFICTECH,28519,기타 가정용 전기 기기 제조업
5,COMPANY-AMOSPROFESSIONAL,46443,화장품 및 화장용품 도매업
6,COMPANY-COSRX,20423,화장품 제조업


In [112]:
display(company_info_df.head(10))

,company_id,industry,main_business,listing_status,employee_count,established_date,field_sources
0,COMPANY-INNISFREE,None,None,NaN,None,2009-11-10,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
1,COMPANY-ESPOIR,None,None,NaN,None,2015-01-02,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
2,COMPANY-AMOREPACIFIC,None,None,상장,None,2006-06-01,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}], 'listing_status': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'corp_cls', 'raw_path': None}]}"
3,COMPANY-ETUDE,None,None,NaN,None,1966-02-19,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
4,COMPANY-PACIFICTECH,None,None,NaN,None,2023-07-27,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
5,COMPANY-AMOSPROFESSIONAL,None,None,NaN,None,1976-01-24,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
6,COMPANY-COSRX,None,None,NaN,None,2013-12-11,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"


In [113]:
# 기업 ID별로 확인한 업종명 가져오기
industry_by_company = overview_df.set_index("company_id")["industry"]

# 현재 보고 있는 company_info_df에 반영
company_info_df["industry"] = (
    company_info_df["company_id"].map(industry_by_company)
)

# 반영 결과 확인
display(company_info_df[["company_id", "industry"]])

,company_id,industry
0,COMPANY-INNISFREE,화장품 및 화장용품 도매업
1,COMPANY-ESPOIR,화장품 및 화장용품 도매업
2,COMPANY-AMOREPACIFIC,화장품 제조업
3,COMPANY-ETUDE,화장품 및 화장용품 도매업
4,COMPANY-PACIFICTECH,기타 가정용 전기 기기 제조업
5,COMPANY-AMOSPROFESSIONAL,화장품 및 화장용품 도매업
6,COMPANY-COSRX,화장품 제조업


In [114]:
display(company_info_df.head(10))

,company_id,industry,main_business,listing_status,employee_count,established_date,field_sources
0,COMPANY-INNISFREE,화장품 및 화장용품 도매업,None,NaN,None,2009-11-10,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
1,COMPANY-ESPOIR,화장품 및 화장용품 도매업,None,NaN,None,2015-01-02,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
2,COMPANY-AMOREPACIFIC,화장품 제조업,None,상장,None,2006-06-01,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}], 'listing_status': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'corp_cls', 'raw_path': None}]}"
3,COMPANY-ETUDE,화장품 및 화장용품 도매업,None,NaN,None,1966-02-19,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
4,COMPANY-PACIFICTECH,기타 가정용 전기 기기 제조업,None,NaN,None,2023-07-27,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
5,COMPANY-AMOSPROFESSIONAL,화장품 및 화장용품 도매업,None,NaN,None,1976-01-24,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"
6,COMPANY-COSRX,화장품 제조업,None,NaN,None,2013-12-11,"{'established_date': [{'source_type': 'public_data', 'source_name': 'Open DART 기업개황 API', 'title': '기업개황 조회', 'original_url': 'https://opendart.fss.or.kr/api/company.json', 'published_at': None, 'collected_at': None, 'as_of_date': None, 'evidence_location': 'est_dt', 'raw_path': None}]}"


In [115]:
# 2025년 사업보고서의 직원 현황 조회
employee_results = []
employee_status = []

for company in companies:
    response = requests.get(
        "https://opendart.fss.or.kr/api/empSttus.json",
        params={
            "crtfc_key": dart_api_key,
            "corp_code": company["dart_corp_code"],
            "bsns_year": "2025",
            "reprt_code": "11011",  # 사업보고서
        },
        timeout=30,
    )

    if response.status_code != 200:
        employee_status.append({
            "기업명": company["input_name"],
            "조회 결과": f"HTTP 오류 {response.status_code}",
        })
        continue

    data = response.json()

    if data.get("status") != "000":
        employee_status.append({
            "기업명": company["input_name"],
            "조회 결과": f'{data.get("status")} - {data.get("message")}',
        })
        continue

    items = data.get("list", [])

    for item in items:
        employee_results.append({
            "company_id": company["company_id"],
            "기업명": company["input_name"],
            **item,
        })

    employee_status.append({
        "기업명": company["input_name"],
        "조회 결과": f"성공 · {len(items)}개 항목",
    })

print("기업별 조회 결과")
display(pd.DataFrame(employee_status))

print("직원 현황 API 응답")
employee_df = pd.DataFrame(employee_results)
display(employee_df.head(10))

기업별 조회 결과


,기업명,조회 결과
0,이니스프리,013 - 조회된 데이타가 없습니다.
1,에스쁘아,013 - 조회된 데이타가 없습니다.
2,아모레퍼시픽,성공 · 4개 항목
3,에뛰드,013 - 조회된 데이타가 없습니다.
4,퍼시픽테크,013 - 조회된 데이타가 없습니다.
5,아모스프로페셔널,013 - 조회된 데이타가 없습니다.
6,코스알엑스,013 - 조회된 데이타가 없습니다.


직원 현황 API 응답


,company_id,기업명,rcept_no,corp_cls,corp_code,corp_name,sexdstn,fo_bbm,reform_bfe_emp_co_rgllbr,reform_bfe_emp_co_cnttk,reform_bfe_emp_co_etc,rgllbr_co,rgllbr_abacpt_labrr_co,cnttk_co,cnttk_abacpt_labrr_co,sm,avrg_cnwk_sdytrn,fyer_salary_totamt,jan_salary_am,rm,stlm_dt
0,COMPANY-AMOREPACIFIC,아모레퍼시픽,20260318000785,Y,00583424,아모레퍼시픽,남,화장품/생활용품,-,-,-,592,-,11,-,603,14.0,"74,187,000,000","129,000,000",-,2025-12-31
1,COMPANY-AMOREPACIFIC,아모레퍼시픽,20260318000785,Y,00583424,아모레퍼시픽,여,화장품/생활용품,-,-,-,"1,732",71,184,4,"1,916",13.0,"137,946,000,000","81,000,000",-,2025-12-31
2,COMPANY-AMOREPACIFIC,아모레퍼시픽,20260318000785,Y,00583424,아모레퍼시픽,남,지원/R&D/SCM,-,-,-,"1,069",-,28,-,"1,097",14.1,"124,487,000,000","126,000,000",-,2025-12-31
3,COMPANY-AMOREPACIFIC,아모레퍼시픽,20260318000785,Y,00583424,아모레퍼시픽,여,지원/R&D/SCM,-,-,-,"1,038",28,94,-,"1,132",14.4,"101,562,000,000","101,000,000",-,2025-12-31


In [116]:
import os
from dotenv import load_dotenv

# 수정한 .env 파일을 다시 읽습니다.
load_dotenv("/Users/anna/bulsae/.env", override=True)

# 공공데이터포털 인증키를 가져옵니다.
data_go_kr_key = os.getenv("DATA_GO_KR_API_KEY")

# 인증키 자체는 출력하지 않고, 등록 여부만 확인합니다.
if data_go_kr_key:
    print("공공데이터포털 인증키가 등록되었습니다.")
else:
    print(".env 파일에 DATA_GO_KR_API_KEY가 있는지 확인해 주세요.")

공공데이터포털 인증키가 등록되었습니다.


In [117]:
import requests
import pandas as pd

# 금융위원회 기업기본정보 API의 기업개요 조회 주소입니다.
url = (
    "https://apis.data.go.kr/1160100/service/"
    "GetCorpBasicInfoService_V2/getCorpOutline_V2"
)

# API에 전달할 요청 조건입니다.
params = {
    "serviceKey": data_go_kr_key,  # .env에서 읽은 인증키
    "resultType": "json",         # JSON 형식으로 응답받기
    "pageNo": 1,                  # 첫 번째 페이지
    "numOfRows": 10,              # 최대 10개
    "corpNm": "에뛰드",            # 조회할 기업명
}

# 인증키가 포함된 URL이나 params는 출력하지 않습니다.
try:
    response = requests.get(url, params=params, timeout=30)
except requests.RequestException:
    print("API 연결에 실패했습니다. 인터넷 연결을 확인해 주세요.")
else:
    print("HTTP 응답 상태:", response.status_code)

    if response.status_code != 200:
        print("API 요청에 실패했습니다. 응답 상태를 확인해 주세요.")
    else:
        try:
            data = response.json()
        except ValueError:
            print("JSON 응답을 받지 못했습니다.")
            print("활용신청 승인 상태와 인증키를 확인해 주세요.")
        else:
            # API 응답에서 처리 결과와 데이터 본문을 꺼냅니다.
            api_response = data.get("response", {})
            header = api_response.get("header", {})
            body = api_response.get("body", {})

            print("결과 코드:", header.get("resultCode"))
            print("결과 메시지:", header.get("resultMsg"))

            # 00은 정상 처리 결과입니다.
            if header.get("resultCode") != "00":
                print("서비스 활용신청 상태와 인증키를 확인해 주세요.")
            else:
                # 결과가 없을 때도 오류가 나지 않도록 처리합니다.
                items_container = body.get("items") or {}
                items = items_container.get("item") or []

                # 한 건일 때 딕셔너리로 반환되면 리스트로 바꿉니다.
                if isinstance(items, dict):
                    items = [items]

                print("조회된 전체 건수:", body.get("totalCount", 0))

                if not items:
                    print("에뛰드로 조회된 기업개요가 없습니다.")
                else:
                    public_company_df = pd.DataFrame(items)

                    # 먼저 어떤 컬럼이 제공되는지 이름만 확인합니다.
                    print("제공된 컬럼:")
                    print(public_company_df.columns.tolist())

HTTP 응답 상태: 403
API 요청에 실패했습니다. 응답 상태를 확인해 주세요.
